In [1]:
import spacy
import os
from Bio import Entrez
import pandas as pd
import ssl
import import_ipynb
import lxml
import genetic_extr as gen
ssl._create_default_https_context = ssl._create_unverified_context

Saved XML


<string>:1: SyntaxWarning: invalid escape sequence '\.'
c:\Users\nicol\OneDrive\Documents\Capstone\.venv\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


                                                text   label
0                                            DOCTYPE  ENTITY
1                                             PUBLIC  ENTITY
2                                         Owner="NLM  ENTITY
3                         IssnType="Electronic">2574  ENTITY
4  CitedMedium="Internet"><Volume>9</Volume><Issu...  ENTITY


In [ ]:
#1. Download papers from PubMed Central

#Parse each paper properly
from bs4 import BeautifulSoup

papers = gen.get_variant_articles_ncbi_only("BRCA1")

soup = BeautifulSoup(papers, "lxml")

papers_list = []

articles = soup.find_all("article")

for article in articles:
    title = article.find("article-title")
    abstract = article.find("abstract")

    papers_list.append({
        "title": title.get_text(" ", strip=True) if title else "",
        "abstract": abstract.get_text(" ", strip=True) if abstract else ""
    })

C:\Users\nicol\AppData\Local\Temp\ipykernel_27424\136950715.py:10: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(papers, "lxml")


In [3]:
#2. Load models (scispaCy + BioNLP13CG)

# Base scientific model
nlp_sci = spacy.load("en_core_sci_sm")

# BioNLP model (biomedical entities)
nlp_bionlp = spacy.load("en_ner_bionlp13cg_md")



In [ ]:
#4. Extract biomedical entities (BioNLP13CG + scispaCy) paper by paper
import pandas as pd

def extract_genes_and_diseases(papers_list):
    extracted_data = []

    for i, paper in enumerate(papers_list):
        abstract_text = paper.get("abstract", "")
        if not abstract_text:
            continue

        doc = nlp_bionlp(abstract_text)

        # Filters for gene and disease
        for ent in doc.ents:
            category = None
            if ent.label_ == "GENE_OR_GENE_PRODUCT":
                category = "Gene"
            elif ent.label_ in ["CANCER", "PATHOLOGICAL_FORMATION"]:
                category = "Disease"

            if category:
                extracted_data.append({
                    "PMID_Index": i,
                    "Term": ent.text.strip(),
                    "Category": category,
                    "Start_Char": ent.start_char,
                    "End_Char": ent.end_char
                })
    # DataFrame is created duplicates are removed from the same paper
    df = pd.DataFrame(extracted_data)
    if not df.empty:
        df = df.drop_duplicates(subset=["PMID_Index", "Term", "Category"])
    
    return df


df_entities = extract_genes_and_diseases(papers_list)

df_entities.to_csv("results/gene_and_disease_entities.csv", index=False)

In [12]:
#6. Normalize variants

def normalize_variant(v):
    v = v.strip()

    # Protein normalization
    if v.startswith("p."):
        v = v.replace(" ", "")

    # DNA normalization
    if v.startswith("c."):
        v = v.upper()

    # SNP normalization
    if v.startswith("rs"):
        v = v.lower()

    return v

normalized_variants = [normalize_variant(v) for v in variants]

print("Normalized Variants:", normalized_variants)

NameError: name 'variants' is not defined